# LLM训练详细步骤解析

本文件将逐步演示如何从零开始训练一个简单的语言模型（LLM），并对每一行代码进行详细解释。

In [38]:
# 导入必要的库
# torch: PyTorch深度学习框架，用于构建和训练神经网络
# torch.nn: PyTorch的神经网络模块，包含各种层和损失函数
# torch.nn.functional: 包含各种函数式接口，如激活函数、损失函数等
# os: 操作系统接口，用于文件和目录操作
# requests: 用于发送HTTP请求，下载网络资源
# tiktoken: OpenAI开发的高效分词器，用于将文本转换为token
# pandas: 数据处理和分析库，用于数据操作和展示
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import requests
import tiktoken
import pandas as pd

In [39]:
# 设置训练超参数
# 超参数是模型训练前需要设定的参数，它们不是通过训练学习得到的

# 批次大小：每次训练时处理的样本数量
# 较大的批次可以提高训练效率但需要更多内存
# 批次大小影响梯度估计的稳定性，较大的批次通常能提供更稳定的梯度估计
batch_size = 4

# 上下文长度（序列长度）：每个输入序列包含的token数量
# 也就是模型一次处理的文本长度
# 这个参数决定了模型能够看到多长的历史信息
context_length = 16 # seq_length

# 模型维度：词向量的维度大小
# 也是Transformer模型中各层的隐藏状态维度
# 维度越高，模型表达能力越强，但计算成本也越高
d_model = 512

# 注意力头数：多头注意力机制中注意力头的数量
# 用于并行处理不同子空间的信息
# 多头机制能让模型在不同表示子空间中关注不同位置的信息
num_heads = 8

# 设备选择：如果CUDA可用则使用GPU加速计算，否则使用CPU
# GPU在并行计算方面比CPU更高效，特别适合深度学习训练
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 设置随机种子以确保实验可重现性
# 在深度学习中，很多操作具有随机性（如参数初始化、数据打乱等）
# 设置随机种子可以确保每次运行得到相同的结果
TORCH_SEED = 42
torch.manual_seed(TORCH_SEED)

In [40]:
# 准备数据集
# 注释掉的代码展示了一种从网络下载数据集的方法
# 这里我们使用本地的订单商品名称数据集

# 打开并读取数据文件
# 'data/订单商品名称.csv'是我们的训练数据文件
# read()方法将整个文件内容读取为一个字符串
with open('data/订单商品名称.csv', 'r') as f:
    text = f.read()

In [41]:
# 使用 tiktoken 分词器进行文本分词
# 分词是将文本切分为更小单元（token）的过程
# 在LLM训练中，分词质量直接影响模型性能

# 获取cl100k_base编码器
# 这是OpenAI的tiktoken库中的一种编码器，适用于GPT系列模型
enc = tiktoken.get_encoding("cl100k_base")

# 将文本编码为token序列
# encode方法将字符串转换为整数列表，每个整数代表一个token
tokenized_text = enc.encode(text)

# 计算数据集的基本统计信息
# total_texts: 原始文本的字符总数
total_texts = len(text)

# total_tokens: 分词后的token总数
total_tokens = len(tokenized_text)

# vocab_size: 词汇表大小（不重复的token数量）
vocab_size = len(set(tokenized_text))

# max_token_value: token的最大值
# 这个值决定了词汇表的大小，用于后续创建嵌入层
max_token_value = max(tokenized_text)

# 打印数据集统计信息
print(f"Total texts: {total_texts}")
print(f"Total tokens: {total_tokens}")
print(f"Vocab size: {vocab_size}")
print(f"Max token value: {max_token_value}")

In [42]:
# 使用 integer mapping 分词器
# 这是一种简单的字符级分词方法
# 与tiktoken不同，这种方法将每个唯一字符映射为一个整数

# 获取文本中所有唯一字符并排序
# set()去重，list()转换为列表，sorted()排序
characters = sorted(list(set(text)))

# 创建字符到索引的映射字典
# enumerate()为每个字符分配索引
# i2s: index to string，索引到字符的映射
i2s = {i:ch for i,ch in enumerate(characters)}

# s2i: string to index，字符到索引的映射
s2i = {ch:i for i,ch in enumerate(characters)}

# 定义编码函数
# 将文本字符串转换为整数列表
def encode(texts):
    return [s2i[ch] for ch in texts]

# 定义解码函数
# 将整数列表转换回文本字符串
def decode(integers):
    return ''.join([i2s[i] for i in integers])

# 使用自定义编码器对整个文本进行编码
tokenized_text = encode(text)

# 重新计算最大token值
max_token_value = max(tokenized_text)

# 打印分词结果统计
print(f"Total tokens: {len(tokenized_text)}")
print(f"Vocab size: {max_token_value}")
print(f"Vocab size: {max_token_value}")

In [43]:
# 创建嵌入层示例
# 嵌入层是神经网络中用于将离散的token转换为连续向量的层

# 创建一个嵌入层
# 参数16表示词汇表大小，512表示嵌入维度
embedding = nn.Embedding(16,512)

# 查看嵌入层权重的形状
# 嵌入层的权重是一个矩阵，形状为(词汇表大小, 嵌入维度)
embedding.weight.shape

In [44]:
# 将分词后的文本转换为PyTorch张量
# PyTorch张量是深度学习框架中的基本数据结构

# 将token列表转换为PyTorch张量
# dtype=torch.long表示数据类型为长整型
# device=device表示将张量存储在指定设备（CPU或GPU）上
tokenized_text = torch.tensor(tokenized_text, dtype=torch.long, device=device)

# 划分训练集和验证集
# 通常使用90%的数据作为训练集，10%作为验证集
# split_idx是划分点的索引
split_idx = int(len(tokenized_text) * 0.9)

# 前90%的数据作为训练集
train_data = tokenized_text[:split_idx]

# 后10%的数据作为验证集
val_data = tokenized_text[split_idx:]

In [45]:
# 模拟 x_batch, y_batch 数据集
# 在语言模型训练中，输入和目标通常是同一序列的不同位置
# x_batch是输入序列，y_batch是目标序列（通常是x_batch向后移动一个位置）

# 随机生成批次索引
# 从训练数据中随机选择batch_size个起始位置
# high参数确保不会越界（需要保证有足够的上下文长度）
idxs = torch.randint(low=0, high=len(train_data) - context_length, size=(batch_size,))

# 构建输入批次
# 对于每个索引，取从该位置开始的context_length个token作为输入
x_batch = torch.stack([train_data[idx:idx + context_length] for idx in idxs])

# 构建目标批次
# 对于每个索引，取从该位置+1开始的context_length个token作为目标
# 这样y_batch就是x_batch向后移动一个位置
y_batch = torch.stack([train_data[idx + 1:idx + context_length + 1] for idx in idxs])

# 查看输入批次的形状
# 形状应该是(batch_size, context_length)
x_batch.shape

In [46]:
# 创建token嵌入查找表
# 嵌入层将离散的token索引转换为连续的向量表示

# 创建嵌入层
# max_token_value是词汇表大小
# d_model是嵌入维度
token_embedding_lookup_table = nn.Embedding(max_token_value, d_model)

# 将输入批次转换为嵌入向量
# 通过嵌入层，每个token索引被转换为d_model维的向量
x = token_embedding_lookup_table(x_batch)

# 将目标批次也转换为嵌入向量（用于后续计算损失）
y = token_embedding_lookup_table(y_batch)

# 获取嵌入层权重的初始值
# 用于后续比较训练前后权重的变化
tb_weights_before = token_embedding_lookup_table.get_parameter(target='weight')

In [47]:
# 展示嵌入向量
# 将PyTorch张量转换为Pandas DataFrame便于查看

# 将第一个批次的嵌入向量转换为DataFrame
pd.DataFrame(x[0].detach().cpu().numpy())

# 保存为变量以供后续使用
xx = pd.DataFrame(x[0].detach().cpu().numpy())

# 设置Pandas显示选项
# display.expand_frame_repr=False防止DataFrame被截断显示
pd.set_option('display.expand_frame_repr', False)

# 打印第一个批次的嵌入向量
print("Our batches:
", pd.DataFrame(xx))

In [48]:
# Positional encoding 位置编码
# 由于Transformer模型不包含循环或卷积结构，
# 需要位置编码来为模型提供序列中token位置的信息

# 导入数学库
import math

# 创建位置编码查找表
# 形状为(context_length, d_model)
# 每一行对应一个位置，每一列对应一个维度
position_encoding_lookup_table = torch.zeros(context_length, d_model)

# 创建位置索引
# torch.arange创建从0到context_length-1的序列
# unsqueeze(1)在第二维增加一个维度，形状变为(context_length, 1)
position = torch.arange(0, context_length, dtype=torch.float).unsqueeze(1)

# 计算位置编码的分母项
# 这是Transformer论文中位置编码公式的一部分
# 使用不同的频率来编码不同维度的位置信息
div_term = torch.exp(-math.log(10000.0) * torch.arange(0, d_model, 2, dtype=torch.float) / d_model)

# 填充位置编码表的偶数列
# 使用正弦函数编码位置信息
position_encoding_lookup_table[:, 0::2] = torch.sin(position * div_term)

# 填充位置编码表的奇数列
# 使用余弦函数编码位置信息
position_encoding_lookup_table[:, 1::2] = torch.cos(position * div_term)

# 扩展位置编码表以匹配批次大小
# unsqueeze(0)在第一维增加一个维度
# expand(batch_size, -1, -1)将第一维扩展到batch_size
position_encoding_lookup_table = position_encoding_lookup_table.unsqueeze(0).expand(batch_size, -1, -1)

# 查看位置编码表的形状
position_encoding_lookup_table.shape

In [49]:
# 将位置编码添加到token嵌入中
# 这是Transformer模型的关键步骤
# 通过加法将位置信息注入到token表示中

# 将token嵌入和位置编码相加
# 广播机制会自动处理维度匹配
x = x + position_encoding_lookup_table

In [50]:
# 另外一种位置编码实现方法
# 这是使用NumPy实现的相同位置编码算法

# 导入NumPy库
import numpy as np

# 定义位置编码函数
def positionalEncoding(length, d, n=10000):
    # 创建一个全零矩阵
    P = np.zeros((length, d))
    
    # 遍历每个位置
    for j in range(length):
        # 遍历每个维度（步长为2，处理成对的维度）
        for i in np.arange(int(d/2)):
            # 计算分母项
            denominator = np.power(n, 2*i/d)
            
            # 偶数维度使用正弦函数
            P[j, 2*i] = np.sin(j/denominator)
            
            # 奇数维度使用余弦函数
            P[j, 2*i+1] = np.cos(j/denominator)
    
    return P

# 定义可视化函数
def visualize_pe(pe):
    # 使用matplotlib绘制热力图
    plt.imshow(pe, aspect="auto")
    
    # 设置标题和轴标签
    plt.title("Positional Encoding")
    plt.xlabel("Encoding Dimension")
    plt.ylabel("Position Index")
    
    # 添加颜色条
    plt.colorbar()
    
    # 显示图像
    plt.show()

# 可视化位置编码
visualize_pe(positionalEncoding(context_length, d_model))

In [51]:
# 可视化位置编码（使用PyTorch实现的结果）

# 导入matplotlib库
import matplotlib.pyplot as plt

# 定义可视化函数
def visualize_pe(pe):
    # 使用matplotlib绘制热力图
    plt.imshow(pe, aspect="auto")
    
    # 设置标题和轴标签
    plt.title("Positional Encoding")
    plt.xlabel("Encoding Dimension")
    plt.ylabel("Position Index")
    
    # 添加颜色条
    plt.colorbar()
    
    # 显示图像
    plt.show()

# 可视化PyTorch实现的位置编码
# .cpu().numpy()将GPU上的张量转换为CPU上的NumPy数组
visualize_pe(position_encoding_lookup_table[0].cpu().numpy())

In [52]:
# Multi-head attention 多头注意力机制
# 注意：这里的实现是跟论文中的实现不一样的
# 论文实际上是为每个头做了Wq、Wk、Wv
# 我们在手写第二期时使用论文中方法

# 创建注意力机制的线性变换层
# Wq: Query权重矩阵
Wq = nn.Linear(d_model, d_model)

# Wk: Key权重矩阵
Wk = nn.Linear(d_model, d_model)

# Wv: Value权重矩阵
Wv = nn.Linear(d_model, d_model)

# 对输入序列进行线性变换
# 生成Query、Key、Value矩阵
Q = Wq(x)
K = Wk(x)
V = Wv(x)

# 重塑和转置以适应多头注意力
# 将d_model维度分解为num_heads和每个头的维度
# reshape: 重新塑造张量形状
# permute: 重新排列维度顺序
Q = Q.reshape(batch_size, context_length, num_heads, -1).permute(0, 2, 1, 3)
K = K.reshape(batch_size, context_length, num_heads, -1).permute(0, 2, 1, 3)
V = V.reshape(batch_size, context_length, num_heads, -1).permute(0, 2, 1, 3)

# 查看Q和K转置的形状
Q.shape, K.transpose(-2, -1).shape

In [53]:
# 计算注意力分数
# 通过Query和Key的点积计算注意力分数
# 这表示每个位置对其他位置的关注程度

# 矩阵乘法计算注意力分数
# @是PyTorch中的矩阵乘法操作符
attn = Q @ K.transpose(-2, -1)

# 查看注意力分数的形状
attn.shape

In [54]:
# 展示注意力分数矩阵
# 将第一个批次第一个头的注意力分数转换为DataFrame并显示
pd.DataFrame(attn[0][0].detach().cpu().numpy())

In [55]:
# 缩放注意力分数
# 为了避免大数值导致softmax函数梯度消失
# 需要将注意力分数除以sqrt(d_k)，其中d_k是每个头的维度

# 缩放注意力分数
attn = attn / math.sqrt(d_model // num_heads)

# 展示缩放后的注意力分数
pd.DataFrame(attn[0][0].detach().cpu().numpy())

In [56]:
# 可视化注意力分数
# 使用matplotlib绘制注意力分数的热力图

# 绘制第一个批次第一个头的注意力分数
plt.imshow(attn[0, 0].detach().cpu().numpy(), "Accent", aspect="auto")

# 设置标题
plt.title("Attention(Q,K)")

# 添加颜色条
plt.colorbar()

In [57]:
# Masking 掩码机制
# 在语言模型训练中，为了防止模型看到未来的信息
# 需要使用掩码机制屏蔽未来位置的信息

# 创建掩码矩阵
# torch.ones创建全1矩阵
mask = torch.ones(attn.shape[-2:])

# 创建上三角矩阵（对角线及以上为1）
# diagonal=1表示从主对角线上方开始
mask = torch.triu(mask, diagonal=1).bool()

# 应用掩码
# 将掩码位置的值替换为负无穷
# 在后续softmax操作中，负无穷会变为0
attn = attn.masked_fill(mask, float("-inf"))

# 展示应用掩码后的注意力分数
pd.DataFrame(attn[0][0].detach().cpu().numpy())

In [58]:
# 可视化应用掩码后的注意力分数

# 绘制第三个批次第四个头的注意力分数
plt.imshow(attn[2, 3].detach().cpu().numpy(), "Accent", aspect="auto")

# 设置标题
plt.title("Attention(Q,K)")

# 添加颜色条
plt.colorbar()

In [59]:
# 应用Softmax函数
# 将注意力分数转换为概率分布
# dim=-1表示在最后一个维度上应用softmax

# 对注意力分数应用softmax
attn = torch.softmax(attn, dim=-1)

In [60]:
# 计算加权Value
# 使用注意力权重对Value进行加权求和
# 得到每个位置的上下文表示

# 矩阵乘法计算加权Value
A = attn @ V

# 查看加权Value的形状
A.shape

In [61]:
# Concatenate 合并多头
# 将多个注意力头的结果合并

# 调整维度顺序
A = A.transpose(1, 2)

# 重塑张量形状
# 将多头维度合并回d_model维度
A = A.reshape(batch_size, context_length, d_model)

In [62]:
# Visualize the attention score using BertViz
# 使用BertViz库可视化注意力分数

# 导入BertViz库
from bertviz import head_view, model_view

# 准备注意力权重数据
# 为每个批次创建单独的注意力权重张量
attention_weights_first_head = [attn[i].unsqueeze(0) for i in range(batch_size)]

# 创建token列表
# 为head_view函数准备所需的token格式
tokens_list = [[decode([idx])] for idx in x_batch[0].tolist()]

# 使用head_view可视化注意力
head_view(attention_weights_first_head, tokens_list, prettify_tokens=False)

In [63]:
# 创建输出投影层
# 将多头注意力的输出投影回原始维度

# 创建线性层
Wo = nn.Linear(d_model, d_model)

# 应用线性变换
output = Wo(A)

In [64]:
# 查看输出投影层的参数
# named_parameters()返回层中所有参数的名称和值

for name, value in Wo.named_parameters():
    print(name, value)

In [65]:
# residual connection 残差连接
# 将注意力输出与原始输入相加
# 残差连接有助于缓解深度网络中的梯度消失问题

output = output + x

# Layer Normalization 层归一化
# 对每个样本的特征进行归一化
# 有助于稳定训练过程

layer_norm1 = nn.LayerNorm(d_model)
output2 = layer_norm1(output)

In [66]:
# 前馈网络（Feed Forward Network）
# Transformer中的前馈网络是一个简单的全连接网络

# 创建第一个线性层
# 将输入维度扩展4倍
linear1 = nn.Linear(d_model, d_model * 4)

# 应用线性变换
output2 = linear1(output2)

In [68]:
# 应用ReLU激活函数
# ReLU是非线性激活函数，增加模型的表达能力

output2 = nn.ReLU()(output2)

In [69]:
# 创建第二个线性层
# 将维度从扩展后的大小恢复到原始维度
linear2 = nn.Linear(d_model * 4, d_model)

# 应用线性变换
output2 = linear2(output2)

In [70]:
# residual connection 残差连接
# 将前馈网络的输出与输入相加

output = output2 + x

# Layer Normalization
# 对输出进行层归一化

layer_norm2 = nn.LayerNorm(d_model)
output = layer_norm2(output)

In [71]:
# 创建词汇表投影层
# 将模型输出映射到词汇表大小
# 用于预测下一个token的概率分布

linear_vocab = nn.Linear(d_model, max_token_value)
output = linear_vocab(output)

In [72]:
# 计算预测结果
# 获取每个位置最可能的token

# logits是模型的原始输出（未归一化的概率）
logits = output

# 获取第一个批次第一个位置的预测结果
# argmax返回最大值的索引
torch.argmax(logits[0][0])

In [73]:
# 解码预测结果
# 将token索引转换回字符

# 解码索引72对应的字符
decode([72])

In [74]:
# 查看logits的形状
# 形状应该是(batch_size, context_length, vocab_size)

logits.shape

In [75]:
# 解码输入批次
# 将输入的token序列转换回文本

# 创建空列表存储解码结果
x_batch_decoded = []

# 遍历每个批次
for i in range(batch_size):
    # 解码第i个批次
    x_batch_decoded.append(decode(x_batch[i].tolist()))

# 打印解码结果
x_batch_decoded

In [76]:
# 解码模型预测结果

# 获取所有位置的预测结果
# dim=-1表示在最后一个维度上应用argmax
predicted_indices = torch.argmax(logits, dim=-1)

# 创建空列表存储解码结果
logits_decoded = []

# 遍历每个批次
for i in range(batch_size):
    # 解码第i个批次的预测结果
    logits_decoded.append(decode(predicted_indices[i].tolist()))

# 打印解码结果
logits_decoded

In [77]:
# 再次查看输出投影层的参数
# 用于后续比较训练前后参数的变化

for name, value in Wo.named_parameters():
    print(name, value)

In [78]:
# 计算损失并进行反向传播
# 这是模型训练的核心步骤

# 获取logits和目标的形状
B, C, D = logits.shape

# 重塑logits以适应损失函数
# 将三维张量展平为二维
logits_reshaped = logits.reshape(B * C, D)

# 重塑目标以适应损失函数
# 将二维张量展平为一维
y_batch_reshaped = y_batch.reshape(B * C)

# 计算交叉熵损失
# 交叉熵损失用于衡量预测分布与真实分布之间的差异
loss = F.cross_entropy(input=logits_reshaped, target=y_batch_reshaped)

# 创建优化器
# AdamW是Adam优化器的改进版本，包含权重衰减
optimizer = torch.optim.AdamW(
    # 指定需要优化的参数
    # 包括所有模型组件的参数
    params = list(token_embedding_lookup_table.parameters()) +
            list(Wo.parameters()) +
            list(Wq.parameters()) +
            list(Wk.parameters()) +
            list(Wv.parameters()) +
            list(linear1.parameters()) +
            list(linear2.parameters()) +
            list(linear_vocab.parameters()) +
            list(layer_norm1.parameters()) +
            list(layer_norm2.parameters()),
            # 学习率
            lr=1e-1
)

# 反向传播
# 计算损失相对于所有参数的梯度
loss.backward()

# 打印嵌入层的梯度
# 用于调试和理解模型学习过程
print(token_embedding_lookup_table.weight.grad)

# 更新参数
# 根据计算得到的梯度更新模型参数
optimizer.step()

In [79]:
# 再次查看输出投影层的参数
# 验证参数是否已更新

for name, value in Wo.named_parameters():
    print(name, value)

In [80]:
# 重新计算注意力分数
# 验证训练后注意力机制的变化

# 重新计算Q和K
Q2 = Wq(x).reshape(batch_size, context_length, num_heads, -1).permute(0, 2, 1, 3)
K2 = Wk(x).reshape(batch_size, context_length, num_heads, -1).permute(0, 2, 1, 3)

# 重新计算注意力分数
attn2 = Q2 @ K2.transpose(-2, -1) / math.sqrt(d_model // num_heads)

# 应用掩码
attn2 = attn2.masked_fill(mask, float("-inf"))

# 应用softmax
attn2 = torch.softmax(attn2, dim=-1)

In [84]:
# 可视化训练后的注意力分数

# 绘制第一个批次第一个头的注意力分数
plt.imshow(attn2[0, 0].detach().cpu().numpy(), "Accent", aspect="auto")

# 设置标题
plt.title("Attention(Q,K)")

# 添加颜色条
plt.colorbar()

In [82]:
# 使用BertViz可视化训练后的注意力分数

# 准备注意力权重数据
attention_weights_first_head2 = [attn2[i].unsqueeze(0) for i in range(batch_size)]

# 创建token列表
tokens_list2 = [[decode([idx])] for idx in x_batch[0].tolist()]

# 使用model_view可视化注意力
model_view(attention_weights_first_head2, tokens_list2, prettify_tokens=False)

In [83]:
# 计算模型总参数数量
# 用于评估模型的复杂度

# 计算各个组件的参数数量并求和
total_params = (max_token_value)*d_model + 4*d_model*d_model + 2*d_model + d_model*4*d_model + 4*d_model*d_model + d_model*max_token_value

# 打印总参数数量
total_params